# Cluster facts read from the logs

## Config

In [15]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from logstore.db import connect

conn = connect()

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


def read(sql):
    return pd.read_sql(sql, conn)

## Kinds of source in the node column

In [16]:
sources = read("SELECT node, component, COUNT(*) AS n_logs FROM logs GROUP BY node, component")
# the source with every number replaced by N, e.g. node-25 -> node-N
sources["kind"] = sources.node.str.replace(r"\d+", "X", regex=True)
by_kind = sources.groupby("kind")
pd.DataFrame({
    "sources": by_kind.node.nunique(),
    "n_logs": by_kind.n_logs.sum(),
    "example": by_kind.node.min(),
    "components": by_kind.component.agg(lambda c: ", ".join(sorted(set(c)))),
}).sort_values("n_logs", ascending=False)

,sources,n_logs,example,components
kind,,,,
node-X,256,158466,node-0,"action, node, unix.hw"
Interconnect-XNX,8,103383,Interconnect-0N00,switch_module
gigeX,8,98809,gige1,gige
Interconnect-XTX,8,23192,Interconnect-0T00,switch_module
node-DX,8,16907,node-D0,"action, clusterfilesystem, domain"
full,1,5850,full,partition
X,2078,4302,1873,"boot_cmd, del_member_cmd, install_cmd, shutdown_cmd"
node--tcX,8,362,node--tc1,tserver
node-msX,1,120,node-ms0,"node, unix.hw"


## Compute node numbers

In [17]:
node_numbers = sources.node.str.extract(r"^node-(\d+)$")[0].dropna().astype(int)
print(f"{node_numbers.nunique()} compute nodes, numbered {node_numbers.min()} to {node_numbers.max()}")

256 compute nodes, numbered 0 to 255


## Which nodes belong to which domain: operator commands that name one domain and one node

In [18]:
targeting = read("SELECT message FROM logs WHERE message LIKE 'Targeting domains%'")
pattern = r"^Targeting domains:node-D(?P<domain>\d+) and nodes:node-(?P<node>\d+)(?: child of command \d+)?$"
one_node = targeting.message.str.extract(pattern).dropna().astype(int)
print(f"{len(one_node):,} of {len(targeting):,} Targeting messages name one domain and one node")
print(f"domain = node // 32 in {(one_node.domain == one_node.node // 32).sum():,} of them")
one_node.groupby("domain").node.agg(["min", "max", "nunique"])

652 of 2,075 Targeting messages name one domain and one node
domain = node // 32 in 652 of them


,min,max,nunique
domain,,,
0,0,31,32
1,32,61,22
2,64,94,24
3,96,127,25
4,128,158,24
5,160,191,24
6,192,223,30
7,224,255,29


## Commands that name a whole domain

In [19]:
whole = targeting.message.str.extract(r"^Targeting domains:node-D(?P<domain>\d+) and nodes:node-\[(?P<first>\d+)-(?P<last>\d+)\]").dropna().astype(int)
whole = whole[whole["last"] - whole["first"] == 31]
whole.value_counts().rename("commands").reset_index().sort_values("domain", ignore_index=True)

,domain,first,last,commands
0,0,0,31,65
1,1,32,63,56
2,2,64,95,49
3,3,96,127,47
4,4,128,159,46
5,5,160,191,51
6,6,192,223,53
7,7,224,255,55


## Filesystem messages: file domain families and who reports them

In [20]:
fs = read("SELECT node AS reporter, state, message FROM logs WHERE component = 'clusterfilesystem'")
fs["file_domain"] = fs.message.str.extract(r"(?:occurred on |domain (?!panic))(\S+)")[0]
fs["family"] = fs.file_domain.str.replace(r"\d+", "N", regex=True)
fs["server"] = fs.message.str.extract(r"served by node node-(\d+)$")[0].astype(float)
fs["reporter_domain"] = fs.reporter.str.extract(r"^node-D(\d)$")[0].astype(float)
print(f"reporters: {', '.join(repr(r) for r in sorted(fs.reporter.unique()))}")
fs.groupby("family").agg(
    n_logs=("message", "size"), file_domains=("file_domain", "nunique"), reporters=("reporter", "nunique"), servers=("server", "nunique")
).sort_values("n_logs", ascending=False)

reporters: '', 'node-D0', 'node-D1', 'node-D2', 'node-D3', 'node-D4', 'node-D5', 'node-D6', 'node-D7'


,n_logs,file_domains,reporters,servers
family,,,,
storageN,5584,256,2,64
rootN_local,4102,32,8,249
rootN_domain,2212,31,8,256
rootN_tmp,1728,32,8,249
cluster_root_backup,334,1,8,8
cluster_usr_backup,314,1,8,8
cluster_var_backup,300,1,8,8
cluster_root,168,1,8,0
rootN_domain_backup,168,3,8,11


## Is the serving node always in the domain that reports it?

In [21]:
served = fs[fs.server.notna()].copy()
# position of the serving node inside its domain: 0 is the domain's first node
served["position"] = served.server - 32 * served.reporter_domain
print(f"{len(served):,} 'no longer served by node' logs, {served.server.nunique()} distinct serving nodes")
print(f"server in the reporting domain: {(served.server // 32 == served.reporter_domain).sum():,}")

7,982 'no longer served by node' logs, 256 distinct serving nodes
server in the reporting domain: 7,982


## storageN file domains: which nodes serve them, and how the name maps to the node

In [22]:
storage = fs[fs.family == "storageN"].copy()
print(f"{storage.file_domain.nunique()} storage file domains, reported by {sorted(storage.reporter.unique())}")

storage_served = served[served.family == "storageN"].copy()
servers = sorted(storage_served.server.astype(int).unique().tolist())
print(f"{len(servers)} serving nodes: {servers}")
print(f"file domains per serving node: {storage_served.groupby('server').file_domain.nunique().value_counts().to_dict()}")

# storage<g><s><d>: group 1 to 16, server 1 to 4 inside the group, disk 1 to 4
parts = storage_served.file_domain.str.extract(r"^storage(?P<g>\d+)(?P<s>\d)(?P<d>\d)$").astype(int)
expected = (parts.g - 1) * 4 + (parts.s - 1) + 192 * (parts.g > 8)
print(f"groups {parts.g.min()} to {parts.g.max()}, servers {parts.s.min()} to {parts.s.max()}, disks {parts.d.min()} to {parts.d.max()}")
print(f"name gives the serving node in {(expected == storage_served.server).sum()} of {len(storage_served)} logs")
storage_served[expected != storage_served.server][["file_domain", "server"]]

256 storage file domains, reported by ['node-D0', 'node-D7']
64 serving nodes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255]
file domains per serving node: {4: 63, 6: 1}
groups 1 to 16, servers 1 to 4, disks 1 to 4
name gives the serving node in 668 of 670 logs


,file_domain,server
3573,storage142,1.0
3574,storage141,1.0


## rootN file domains: is rootN served by node N-1 of the domain?

In [23]:
root = served[served.file_domain.str.match(r"root\d+_")].copy()
root["member"] = root.file_domain.str.extract(r"^root(\d+)_")[0].astype(int)
root["own_node"] = root.member - 1 == root.position
by_family = root.groupby("family")
pd.DataFrame({
    "n_logs": by_family.size(),
    "members": by_family.member.agg(lambda m: f"{m.min()}-{m.max()}"),
    "served_by_own_node": by_family.own_node.sum(),
    "served_by_first_node_of_domain": by_family.position.agg(lambda p: (p == 0).sum()),
})

,n_logs,members,served_by_own_node,served_by_first_node_of_domain
family,,,,
rootN_domain,2212,2-32,1936,276
rootN_domain_backup,168,1-30,168,163
rootN_local,1742,1-32,1742,4
rootN_local_backup,158,1-30,158,155
rootN_tmp,1718,1-32,1718,2
rootN_tmp_backup,153,1-30,153,150


## The other file domains: position of the serving node inside its domain

In [24]:
shared = served[~served.family.isin(root.family.unique()) & (served.family != "storageN")]
shared.groupby(["family", "position"]).size().unstack(fill_value=0)

position,0.0,1.0,27.0
family,,,
cluster_root_backup,334,0,0
cluster_usr_backup,314,0,0
cluster_var_backup,300,0,0
domain_dskNa,8,0,0
domain_dskNe,0,0,1
root_domain,126,0,0
sc_cluster_backup,61,1,0
usr_domain,15,0,0
var_domain,1,0,0


## Interconnect switch modules: logs per state

In [25]:
switches = read("SELECT node AS module, state, COUNT(*) AS n_logs FROM logs WHERE component = 'switch_module' GROUP BY node, state")
print(f"{switches.module.nunique()} switch modules")
switches.pivot(index="module", columns="state", values="n_logs").fillna(0).astype(int)

16 switch modules


state,bcast-error,control,error,fan,psu,temphigh
module,,,,,,
Interconnect-0N00,0,51,7578,21559,7,120
Interconnect-0N01,36,30,6874,0,7,51
Interconnect-0N02,1,30,5602,0,0,110
Interconnect-0N03,5,30,8382,0,0,266
Interconnect-0T00,1074,35,186,0,6,0
Interconnect-0T01,0,31,109,0,6,0
Interconnect-0T02,0,33,29,0,7,0
Interconnect-0T03,0,38,41,0,7,120
Interconnect-1N00,7,30,10590,0,0,0


## Networks and interfaces named in the NIFF messages

In [26]:
niff = read("SELECT node, message FROM logs WHERE component = 'unix.hw' AND state LIKE 'net.niff%'")
niff[["network", "interface"]] = niff.message.str.extract(r"on network (\S+) via interface (\S+)")
niff.groupby(["interface", "network"]).agg(n_logs=("message", "size"), nodes=("node", "nunique"))

n_logs  nodes
interface network                  
alt0      0.0.0.0       1011    256
          5.5.0.0          6      1
          5.5.224.0     8183    256
          5.5.226.0     3280    254
ee0       0.0.0.0          1      1
          10.128.0.0   10447    253
scip0     10.96.0.0      742    255

## Ethernet switches, terminal servers, the management server and the partition

In [27]:
other = read(
    """
    SELECT node AS source, component, state, COUNT(*) AS n_logs FROM logs
    WHERE component IN ('gige', 'tserver', 'partition') OR node LIKE 'node-ms0%' OR node = 'node-C0'
    GROUP BY node, component, state
    """
)
other["kind"] = other.source.str.replace(r"\d+$", "X", regex=True)
other.groupby(["kind", "component", "state"]).agg(sources=("source", "nunique"), n_logs=("n_logs", "sum"))

sources  n_logs
kind       component state                                    
full       partition status                          1    5850
gigeX      gige      fan.failed                      2       2
                     temperature                     8   98807
node--tcX  tserver   status                          8     362
node-CX    domain    status                          1       1
node-ms0-C domain    status                          1       3
node-msX   node      status                          1       8
                     temperature                     1      38
           unix.hw   state_change.available          1      37
                     state_change.unavailable        1      37